# Databricks SQL Examples

This notebook covers common Databricks SQL patterns:

- querying managed tables with SQL
- creating views
- aggregations for dashboard-style metrics
- window functions
- SQL warehouse usage concepts

## SQL warehouse context

In Databricks, SQL workloads are commonly run through SQL warehouses. They are optimized for BI, dashboards, and analyst-style querying.

In [ ]:
sales_data = [
    (1, "north", "laptop", 1200.0, "2026-04-01"),
    (2, "north", "mouse", 25.0, "2026-04-01"),
    (3, "south", "monitor", 400.0, "2026-04-01"),
    (4, "west", "desk", 300.0, "2026-04-02"),
    (5, "south", "chair", 150.0, "2026-04-02"),
    (6, "north", "monitor", 450.0, "2026-04-02")
]

sales_df = spark.createDataFrame(
    sales_data,
    ["order_id", "region", "product", "amount", "order_date"]
)

target_table = "main.demo.sales_orders_sql"
sales_df.write.format("delta").mode("overwrite").saveAsTable(target_table)
print(f"Prepared table {target_table}")

## Basic SQL query

Use SQL when you want a concise and familiar way to explore and aggregate curated data.

In [ ]:
basic_query = spark.sql("""
SELECT region, product, amount, order_date
FROM main.demo.sales_orders_sql
ORDER BY order_date, region
""")

display(basic_query)

## Aggregations for reporting

This is the type of query commonly used behind dashboards and KPI reporting.

In [ ]:
regional_summary = spark.sql("""
SELECT
  region,
  COUNT(*) AS order_count,
  SUM(amount) AS total_sales,
  AVG(amount) AS avg_order_value
FROM main.demo.sales_orders_sql
GROUP BY region
ORDER BY total_sales DESC
""")

display(regional_summary)

## Create a view

Views are useful for reusing common SQL logic and exposing consistent business-friendly query surfaces.

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW main.demo.sales_orders_daily_summary AS
SELECT
  order_date,
  region,
  COUNT(*) AS order_count,
  SUM(amount) AS total_sales
FROM main.demo.sales_orders_sql
GROUP BY order_date, region
""")

display(spark.sql("SELECT * FROM main.demo.sales_orders_daily_summary ORDER BY order_date, region"))

## Window function example

Window functions are useful when you need rankings, running totals, or comparisons within a group.

In [ ]:
windowed_query = spark.sql("""
SELECT
  order_date,
  region,
  product,
  amount,
  RANK() OVER (PARTITION BY region ORDER BY amount DESC) AS sales_rank_in_region
FROM main.demo.sales_orders_sql
ORDER BY region, sales_rank_in_region
""")

display(windowed_query)

## Dashboard-style KPI query

This kind of query is often used as the source for tiles or charts in BI dashboards.

In [ ]:
kpi_query = spark.sql("""
SELECT
  COUNT(*) AS total_orders,
  SUM(amount) AS total_revenue,
  AVG(amount) AS average_order_value,
  COUNT(DISTINCT region) AS active_regions
FROM main.demo.sales_orders_sql
""")

display(kpi_query)

## Practical guidance

- Use SQL warehouses for analyst and BI-oriented workloads
- Keep business logic in views when multiple reports use the same query pattern
- Use fully qualified names for production objects
- Query curated Delta tables instead of raw files for reporting
- Combine SQL with Unity Catalog permissions for governed data access